# ParamLab Python 等效实现 — OLS / Ridge / Lasso / RF + SHAP

对应 `04_ParamLab.html` 的所有功能：
- 4 种回归算法
- K-fold 交叉验证
- Lasso/Ridge 正则化路径
- 真实 SHAP 解释（基于 shap 库）
- 4 算法 Benchmark 对比


In [ ]:
# pip install scikit-learn shap matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("注意：未安装 shap 库，将使用 Permutation Importance 替代")

## 1. 加载数据 + 特征工程

In [ ]:
# 加载莱势明 800 行模拟数据
df = pd.read_csv("../datasets/work_orders.csv").sample(800, random_state=42).reset_index(drop=True)

# 特征工程
df_machines = pd.read_csv("../datasets/machines.csv")
df["machine_type_encoded"] = pd.Categorical(df["machine_type"]).codes
df["priority_num"] = df["priority"].map({"P0": 3, "P1": 2, "P2": 1})
# y = 估算加工时长
y = df["estimated_proc_hours"]
X = df[["qty", "machine_type_encoded", "priority_num"]].copy()
X = pd.concat([X, pd.get_dummies(df["family"], prefix="fam")], axis=1)

print(f"X shape: {X.shape}, y shape: {y.shape}")

## 2. 训练 / 测试集划分 + 标准化

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 3. 训练 4 种回归模型

In [ ]:
models = {
    "OLS": LinearRegression(),
    "Ridge (λ=1.0)": Ridge(alpha=1.0),
    "Lasso (λ=0.1)": Lasso(alpha=0.1),
    "Random Forest (n=50)": RandomForestRegressor(n_estimators=50, random_state=42),
}
results = []
for name, model in models.items():
    if "Random Forest" in name:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train_s, y_train)
        y_pred = model.predict(X_test_s)
    results.append({
        "Model": name,
        "R²(test)": round(r2_score(y_test, y_pred), 4),
        "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred)), 3),
        "MAE": round(mean_absolute_error(y_test, y_pred), 3),
    })
df_results = pd.DataFrame(results)
df_results

## 4. Lasso 正则化路径

In [ ]:
lambdas = np.logspace(-3, 1, 30)
coefs_path = []
for lam in lambdas:
    m = Lasso(alpha=lam, max_iter=10000).fit(X_train_s, y_train)
    coefs_path.append(m.coef_)
coefs_path = np.array(coefs_path)

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(coefs_path.shape[1]):
    ax.plot(lambdas, coefs_path[:, i], label=X.columns[i])
ax.set_xscale("log"); ax.set_xlabel("λ"); ax.set_ylabel("系数 β")
ax.set_title("Lasso 正则化路径"); ax.legend(loc="best", fontsize=8)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
plt.show()

## 5. K-fold 交叉验证

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    X_use = X if "Random Forest" in name else X_train_s
    scores = cross_val_score(model, X_use[:len(y_train)] if "Random Forest" not in name else X.iloc[:len(y_train)], y_train, cv=kf, scoring="r2")
    print(f"{name:30s}  CV R²: {scores.mean():.4f} ± {scores.std():.4f}")

## 6. SHAP 解释（Random Forest）

In [ ]:
if SHAP_AVAILABLE:
    rf = models["Random Forest (n=50)"]
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, feature_names=X.columns, show=True)
else:
    # Permutation Importance 替代
    from sklearn.inspection import permutation_importance
    rf = models["Random Forest (n=50)"]
    result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
    imp_df = pd.DataFrame({
        "Feature": X.columns,
        "Importance": result.importances_mean
    }).sort_values("Importance", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(imp_df["Feature"], imp_df["Importance"])
    ax.set_title("Permutation Importance")
    plt.tight_layout(); plt.show()

## 总结
本 notebook 与 `04_ParamLab.html` 网页版完全对等。网页版面向"零 Python 基础"学员；本 notebook 面向"有 Python 基础"的本科生 / 研究生进行深度学习。

教学建议：
1. 先用 ParamLab 网页版让学员理解概念（10 分钟）
2. 再让学员打开本 notebook 重现结果（30 分钟）
3. 最后让学员修改参数 / 添加特征，提交实验报告
